# QAM基础知识

**QAM Basics**

---

本notebook介绍正交幅度调制（Quadrature Amplitude Modulation, QAM）的基本概念。QAM是现代数字通信系统中最重要的调制方式之一，在WiFi、5G、卫星通信等领域广泛应用。理解QAM的原理对于学习OTFS调制技术至关重要，因为OTFS符号最终以QAM调制的形式在延迟-多普勒域传输。

## 1. 目标 (Objectives)

通过本notebook，您将：

- **理解I/Q正交调制的原理**：掌握如何通过同相（I）和正交（Q）两路信号构建复数信号
- **掌握QAM星座图表示**：理解星座点如何对应比特映射
- **认识16-QAM、64-QAM、256-QAM的差异**：了解阶数与频谱效率的关系
- **理解格雷码映射的优势**：掌握格雷码如何减少误码传播
- **为学习OTFS打下基础**：OTFS在延迟-多普勒域传输的正是QAM符号

## 2. QAM直观理解 (Conceptual Understanding)

### 2.1 I/Q正交调制原理

QAM的核心思想是利用两个正交载波进行调制：

- **同相载波 (In-Phase, I)**：cos(2πf_c t)
- **正交载波 (Quadrature, Q)**：sin(2πf_c t)

这两个载波相位相差90°，因此相互正交。QAM调制将两路独立的基带信号分别调制到这两个载波上，然后叠加发送：

$$s(t) = I(t) \cdot \cos(2\pi f_c t) + Q(t) \cdot \sin(2\pi f_c t)$$

从另一个角度看，s(t)可以表示为复数信号：

$$s(t) = \text{Re}\{(I(t) + jQ(t)) \cdot e^{j2\pi f_c t}\} = \text{Re}\{x(t) \cdot e^{j2\pi f_c t}\}$$

其中x(t) = I(t) + jQ(t)称为**复包络**或**基带信号**，它是一个复数信号，其实部为I路，虚部为Q路。

### 2.2 为什么使用I/Q调制？

I/Q调制之所以广泛使用，原因包括：

1. **频谱效率加倍**：使用两个正交载波，可以在相同带宽内传输两路独立数据
2. **实现简单**：使用数字信号处理即可轻松生成和解调I/Q信号
3. **便于通道补偿**：信道对I/Q两路的影响可以统一用复数增益表示

### 2.3 QAM星座图

QAM调制将比特映射为复数星座点。每个星座点对应一个复数 x = I + jQ，其中：

- **I（实部）**：同相分量
- **Q（虚部）**：正交分量

在复平面上，这些星座点形成特定的图案。对于M-QAM（M阶QAM），星座图包含M个点，每点携带log₂(M)比特信息。

| 调制方式 | 星座点数M | 每符号比特数 | 频谱效率(bit/s/Hz) |
|---------|----------|------------|-------------------|
| 4-QAM   | 4        | 2          | 2                 |
| 16-QAM  | 16       | 4          | 4                 |
| 64-QAM  | 64       | 6          | 6                 |
| 256-QAM | 256      | 8          | 8                 |
| 1024-QAM| 1024     | 10         | 10                |

### 2.4 格雷码映射

格雷码（Gray Code）是一种相邻码元只差一位的编码方式。在QAM中，格雷码映射确保：

- 任意两个相邻的星座点（例如水平或垂直相邻）只相差1比特
- 当噪声导致判决错误时，通常只会造成1比特错误，而不是多个比特错误

例如16-QAM的格雷码映射：

```
比特: 0000 0001 0011 0010
      ↓↓↓↓↓↓↓↓↓↓↓↓↓
星座点 ● ● ● ●
       ● ● ● ●
       ● ● ● ●
       ● ● ● ●
```

## 3. 代码演示：QAM星座图 (Constellation Diagrams)

下面我们创建16-QAM星座图，直观理解QAM调制的结构。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Set up Chinese font support
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

### 3.1 创建16-QAM星座图

In [ ]:
# Define 16-QAM constellation points
# Using Gray-coded mapping for better error performance

def create_qam16():
    """Create 16-QAM constellation points with Gray coding"""
    # 16-QAM: 4 rows x 4 columns
    # Points are at coordinates (-3,-3), (-3,-1), (-1,-3), etc. scaled by 2
    # Actually let's use the standard normalized constellation
    
    # Standard 16-QAM constellation levels (I and Q)
    levels = np.array([-3, -1, 1, 3]) / 3.0  # Normalized levels
    
    # Create all 16 constellation points
    qam16 = []
    bit_labels = []
    
    # Gray code table for 4 bits
    gray_code = ['0000', '0001', '0011', '0010', '0100', '0101', '0111', '0110',
                  '1100', '1101', '1111', '1110', '1010', '1011', '1001', '1000']
    
    for i, level_i in enumerate(levels):
        for j, level_q in enumerate(levels):
            idx = i * 4 + j
            qam16.append(level_i + 1j * level_q)
            bit_labels.append(gray_code[idx])
    
    return np.array(qam16), bit_labels

qam16, bit_labels = create_qam16()

print(f"16-QAM星座点数量: {len(qam16)}")
print(f"每个符号携带比特数: {int(np.log2(len(qam16)))} bit")
print(f"星座点范围: [{qam16.real.min():.2f}, {qam16.real.max():.2f}]")

In [ ]:
# Plot 16-QAM constellation diagram
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(qam16.real, qam16.imag, c='blue', s=100, marker='o', edgecolors='black', linewidths=1)

# Add bit labels to each point
for i, (point, label) in enumerate(zip(qam16, bit_labels)):
    ax.annotate(label, (point.real, point.imag), 
                textcoords="offset points", xytext=(5, 5), 
                ha='left', fontsize=8, color='red')

ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_xlabel('In-Phase (I)', fontsize=12)
ax.set_ylabel('Quadrature (Q)', fontsize=12)
ax.set_title('16-QAM Constellation Diagram (with Gray Code)', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
plt.show()

print("\n观察：16-QAM形成4×4=16个星座点，每个点对应4比特")
print("注意相邻点之间只差1比特（格雷码映射）")

### 3.2 对比16-QAM、64-QAM、256-QAM

In [ ]:
def create_qam(M):
    """Create M-QAM constellation points"""
    # M must be a perfect square
    k = int(np.log2(M))
    n = int(np.sqrt(M))
    
    # Create normalized levels
    levels = np.linspace(-(n-1), (n-1), n) / (n-1)
    
    qam_points = []
    for i in levels:
        for j in levels:
            qam_points.append(i + 1j * j)
    
    return np.array(qam_points)

# Create constellations
qam16 = create_qam(16)
qam64 = create_qam(64)
qam256 = create_qam(256)

print(f"16-QAM: {len(qam16)} points, {int(np.log2(len(qam16)))} bits/symbol")
print(f"64-QAM: {len(qam64)} points, {int(np.log2(len(qam64)))} bits/symbol")
print(f"256-QAM: {len(qam256)} points, {int(np.log2(len(qam256)))} bits/symbol")

In [ ]:
# Plot comparison of 16-QAM, 64-QAM, 256-QAM
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

constellations = [qam16, qam64, qam256]
titles = ['16-QAM\n(4 bits/symbol)', '64-QAM\n(6 bits/symbol)', '256-QAM\n(8 bits/symbol)']
colors = ['blue', 'green', 'red']

for ax, const, title, color in zip(axes, constellations, titles, colors):
    ax.scatter(const.real, const.imag, c=color, s=30, alpha=0.7)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_xlabel('In-Phase (I)', fontsize=11)
    ax.set_ylabel('Quadrature (Q)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("\n对比观察：")
print("  - 16-QAM: 4×4=16点，星座点间隔较大，抗噪声能力强")
print("  - 64-QAM: 8×8=64点，星座点更密集，频谱效率更高但抗噪声能力稍弱")
print("  - 256-QAM: 16×16=256点，星座点非常密集，需要更高的信噪比")

## 4. 噪声对QAM的影响 (Effect of Noise)

在实际信道中，信号会遭受噪声干扰。AWGN（加性高斯白噪声）是最基本的噪声模型。噪声会导致星座点扩散，增加判决错误的可能性。

### 信噪比与误码率

信噪比（SNR）定义为信号功率与噪声功率之比：

$$SNR_{dB} = 10 \log_{10}\left(\frac{E_s}{N_0}\right)$$

其中E_s是符号能量，N_0是噪声功率谱密度。

对于M-QAM，理论误码率（近似）为：

$$P_b \approx \frac{4}{\log_2 M}\left(1 - \frac{1}{\sqrt{M}}\right)Q\left(\sqrt{\frac{3\log_2 M \cdot E_s}{N_0(M-1)}}\right)$$

其中Q(x)是Q函数。

In [ ]:
from scipy import special

def q_function(x):
    """Q-function: Q(x) = 0.5 * erfc(x / sqrt(2))"""
    return 0.5 * special.erfc(x / np.sqrt(2))

def qam_theoretical_ber(M, EbN0_db):
    """Calculate theoretical BER for M-QAM"""
    k = np.log2(M)
    EbN0 = 10 ** (EbN0_db / 10)
    gamma_b = EbN0  # Energy per bit to noise ratio
    
    # Approximate BER formula for square M-QAM
    if M == 4:
        ber = q_function(np.sqrt(2 * gamma_b))
    else:
        ber = (4 / k) * (1 - 1/np.sqrt(M)) * q_function(np.sqrt(3 * k * gamma_b / (M - 1)))
    
    return ber

# Calculate theoretical BER for different modulation orders
EbN0_db = np.arange(0, 25, 0.5)
ber_16 = [qam_theoretical_ber(16, snr) for snr in EbN0_db]
ber_64 = [qam_theoretical_ber(64, snr) for snr in EbN0_db]
ber_256 = [qam_theoretical_ber(256, snr) for snr in EbN0_db]

# Plot BER curves
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(EbN0_db, ber_16, 'b-', linewidth=2, label='16-QAM')
ax.semilogy(EbN0_db, ber_64, 'g-', linewidth=2, label='64-QAM')
ax.semilogy(EbN0_db, ber_256, 'r-', linewidth=2, label='256-QAM')

ax.set_xlabel('Eb/N0 (dB)', fontsize=12)
ax.set_ylabel('Bit Error Rate (BER)', fontsize=12)
ax.set_title('QAM Theoretical BER vs Eb/N0', fontsize=14)
ax.legend()
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(0, 24)
ax.set_ylim(1e-6, 1)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print("\n观察：")
print("  - 要达到相同BER，更高阶QAM需要更高的Eb/N0")
print("  - 例如：BER=10^-3时，16-QAM约需9dB，64-QAM约需14dB，256-QAM约需18dB")
print("  - 这是用抗噪声性能换取频谱效率的权衡")

In [ ]:
# Simulate noise effect on 16-QAM constellation
def add_noise(signal, EbN0_db, bits_per_symbol):
    """Add AWGN to QAM signal given Eb/N0 in dB"""
    EbN0 = 10 ** (EbN0_db / 10)
    # Calculate Es/N0 from Eb/N0
    EsN0 = EbN0 * bits_per_symbol
    # Signal power (assuming unit energy constellation)
    signal_power = np.mean(np.abs(signal) ** 2)
    # Noise variance = signal_power / (Es/N0)
    noise_var = signal_power / EsN0
    # Generate complex Gaussian noise
    noise = np.sqrt(noise_var / 2) * (np.random.randn(len(signal)) + 1j * np.random.randn(len(signal)))
    return signal + noise

# Generate random 16-QAM symbols
np.random.seed(42)
num_symbols = 500
tx_symbols = qam16[np.random.randint(0, len(qam16), num_symbols)]

# Add noise at different SNR levels
EbN0_levels = [10, 15, 20]  # dB
bits_per_symbol = 4

# Plot constellation with noise
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, EbN0 in zip(axes, EbN0_levels):
    rx_symbols = add_noise(tx_symbols, EbN0, bits_per_symbol)
    
    ax.scatter(rx_symbols.real, rx_symbols.imag, c='blue', s=20, alpha=0.6, label='Received')
    ax.scatter(qam16.real, qam16.imag, c='red', s=50, marker='x', linewidths=2, label='Constellation')
    
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_xlabel('In-Phase (I)', fontsize=11)
    ax.set_ylabel('Quadrature (Q)', fontsize=11)
    ax.set_title(f'16-QAM with AWGN (Eb/N0 = {EbN0} dB)', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("\n噪声效果观察：")
print("  - Eb/N0=10dB: 星座点明显扩散，错误概率较高")
print("  - Eb/N0=15dB: 星座点仍有扩散但重叠较少")
print("  - Eb/N0=20dB: 星座点紧密聚集，错误概率很低")

## 5. 关联OTFS：QAM符号是OTFS传输的信息载体

### OTFS调制流程

OTFS（正交时频空）是一种在延迟-多普勒域进行信号处理的调制方案。但最终，无论在哪个域进行处理，OTFS传输的基本单元仍然是QAM符号。

OTFS的完整调制流程：

```
比特流 → QAM映射 → QAM符号网格 → Zak变换 → 时频域 → 脉冲成形 → 信道
```

### OTFS的核心思想

在OTFS中：

1. **信息以QAM符号形式放置在延迟-多普勒网格上**
   - 延迟轴（τ）对应多径延迟
   - 多普勒轴（ν）对应多普勒频移
   - 每个格点承载一个QAM符号

2. **通过二维傅里叶相关变换（Zak变换）将延迟-多普勒域映射到时频域**
   - 变换后每个时频格点携带一个QAM符号的信息

3. **信道在延迟-多普勒域表示为稀疏冲激响应**
   - 每条多径对应一个延迟-多普勒域的冲激
   - 接收端无需复杂均衡，只需在域中匹配滤波

### QAM在OTFS中的角色

| 方面 | 描述 |
|------|------|
| **符号承载** | OTFS在延迟-多普勒网格的每个格点上放置QAM符号 |
| **比特映射** | 每个QAM符号携带log₂(M)比特（16-QAM为4bit，64-QAM为6bit） |
| **频谱效率** | OTFS整体频谱效率由QAM阶数和时频格点数共同决定 |
| **抗噪声** | 高阶QAM（如256-QAM）需要更高SNR，低阶QAM更鲁棒 |

### 与OFDM的对比

| 特性 | OFDM | OTFS |
|------|------|------|
| **子载波映射** | 每个子载波承载一个QAM符号 | 每个延迟-多普勒格点承载一个QAM符号 |
| **处理域** | 时频域（FFT/IFFT） | 延迟-多普勒域（Zak变换） |
| **信道表示** | 时变频率响应 | 时不变等价信道（稀疏冲激响应） |
| **多径处理** | CP + 频域均衡 | 延迟域匹配滤波 |

### 关键洞察

**QAM是OTFS的信息载体**。无论OTFS的二维变换多复杂，最终的数据传输仍然是通过QAM符号完成的。选择合适的QAM调制阶数需要权衡：

- **频谱效率**：更高阶QAM（更多比特/符号）
- **抗噪声性能**：更高阶QAM需要更高SNR
- **实现复杂度**：高阶QAM的判决和映射更复杂

在OTFS系统中，这种权衡直接影响系统的覆盖范围和吞吐量。

## 6. 思考题 (Review Questions)

1. **I/Q调制原理**：解释为什么I/Q调制可以在相同带宽内传输两路独立数据。提示：考虑两个载波cos(2πf_c t)和sin(2πf_c t)的正交性。

2. **星座图距离**：在16-QAM中，相邻星座点之间的欧氏距离是多少？如果星座点能量归一化为1，相邻点距离为d=2/3。请问256-QAM的相邻点距离是多少？这如何解释高阶QAM需要更高SNR？

3. **格雷码优势**：假设使用普通二进制编码（0000, 0001, 0010, 0011...），当接收机错误判决相邻星座点时，会造成多少比特错误？使用格雷码呢？这种差异对系统性能有何影响？

4. **QAM与OTFS**：在OTFS系统中，每个延迟-多普勒格点承载一个QAM符号。如果OTFS使用64-QAM，调制一个大小为M×N的延迟-多普勒网格，请问：
   - 整个网格携带多少比特？
   - 如果信道中存在K条多径，每条多径如何影响QAM符号的接收？

5. **调制阶数选择**：某通信系统要求在郊区（SNR约15dB）支持30Mbps数据速率，在城市（SNR约25dB）支持100Mbps。请为这两个场景选择合适的QAM阶数，并说明理由。假设系统带宽为10MHz。

6. **误码率对比**：利用理论BER公式，比较16-QAM和64-QAM在10dB和15dB Eb/N0下的误码率差异。哪个阶数的QAM更适合高SNR环境？

---

## 总结 (Summary)

本notebook介绍了QAM调制的基础知识：

- **I/Q正交调制**：利用两个正交载波在相同带宽内传输两路数据
- **星座图**：复平面上QAM符号的可视化表示，每个星座点对应比特映射
- **调制阶数**：16-QAM（4bit）、64-QAM（6bit）、256-QAM（8bit）等
- **格雷码映射**：相邻星座点只差1比特，减少误码传播
- **噪声影响**：SNR越高，星座点越紧密，误码率越低
- **OTFS关联**：OTFS在延迟-多普勒域传输的正是QAM符号

这些概念为理解OTFS调制技术中QAM符号的传输提供了基础。